# Paper 1 — Notebook 1 of 4
## Foundation, Dual-Product Validation & Rainfall Detection

**BD-AgroClim** — Land-Fraction and Sub-Grid Relief Controls on Reanalysis Bias

This notebook produces **Tables T1–T4** and the paired working tables that Notebooks 2–4 consume.

**Run order:** NB1 → NB2 → NB3 → NB4. Run this one first.

**What NB1 does**
1. Loads all 7 datasets with the safe loader (correct filenames only)
2. Homogeneity testing (Pettitt + SNHT) on station series — *step 1.1*
3. Pairs stations ↔ nearest NASA POWER cell ↔ ERA5-Land point — *step 1.2*
4. Pooled validation metrics for **both** products — *step 1.3* (T3)
5. Rainfall detection skill POD/FAR/CSI/HSS by threshold — *step 1.4* (T4)
6. Station inventory (T1) + data-source table (T2)
7. Writes intermediate files to `/kaggle/working/` for the next notebooks

**Outputs written:** `paired_era5.parquet`, `paired_power.parquet`, `station_cell_map.csv`,
`T1_station_inventory.csv`, `T2_data_sources.csv`, `T3_pooled_validation.csv`,
`T4_rainfall_detection.csv`, `homogeneity_results.csv`


### Cell 1 — Config, safe loader, and metric helpers
Everything downstream imports from here. The loader hard-codes the correct filenames so a near-duplicate can never creep in.

In [1]:
import os, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 60)

# ----- PATHS -----------------------------------------------------------------
# On Kaggle, point BASE at your dataset folder. Locally, at the upload folder.
# Added '/dataset/' at the end to match the structure in image_fa4435.png
CANDIDATES = [
    '/kaggle/input/datasets/neloypramanik4444/weather-dataset/dataset/', 
    '/kaggle/input/weather-dataset/dataset/', # Fallback Kaggle path
    '/mnt/user-data/uploads/', 
    './'
]
BASE = next((c for c in CANDIDATES if os.path.exists(c)), './')
OUT  = '/kaggle/working/' if os.path.exists('/kaggle/working/') else './out/'
os.makedirs(OUT, exist_ok=True)
print('BASE =', BASE); print('OUT  =', OUT)

# The NASA POWER filename varies; resolve it robustly from its specific subfolder.
def _find_power():
    power_dir = os.path.join(BASE, 'NASA POWER')
    if os.path.exists(power_dir):
        for f in os.listdir(power_dir):
            if f.startswith('NASA_POWER') and f.endswith('.csv'):
                return f"NASA POWER/{f}"
    return 'NASA POWER/NASA_POWER_Climate_Data_Bangladesh_2000_2026.csv'

# Updated with exact subfolder paths based on the Kaggle directory tree
FILES = {
    'power'  : _find_power(),
    'bmd'    : 'BMD data/bmd_clean_daily.csv',
    'meta'   : 'BMD data/station_metadata_final.csv',
    'era5'   : 'era5land_stations_daily/era5land_stations_daily_v2.csv',   # v2 ALWAYS (non-v2 drops 2 stations)
    'terrain': 'C6_Terrain (slope, TPI, TWI)/terrain_full.csv',
    'skill'  : 'C6_Terrain (slope, TPI, TWI)/terrain_skill_merged_v2.csv',
    'yield'  : 'BBS Crop Yield/yield_panel_final.csv',             # _final ALWAYS
}
def load(key, **kw): return pd.read_csv(BASE + FILES[key], **kw)

# ----- STUDY-PERIOD CONSTANTS ------------------------------------------------
STUDY_START = '2000-01-01'
STUDY_END   = '2023-12-31'          # station overlap ends here
FOCAL_FILLED = ['Kutubdia', 'Sandwip']   # held out of ML training (island focal-fill)
RAIN_THRESHOLDS = [1, 10, 20, 50]        # mm/day for detection skill

# ----- METRIC HELPERS --------------------------------------------------------
def kge(sim, obs):
    sim=np.asarray(sim,float); obs=np.asarray(obs,float)
    m=np.isfinite(sim)&np.isfinite(obs); sim,obs=sim[m],obs[m]
    if len(obs)<3: return np.nan
    r=np.corrcoef(sim,obs)[0,1]
    a=sim.std()/obs.std() if obs.std() else np.nan
    b=sim.mean()/obs.mean() if obs.mean() else np.nan
    return 1-np.sqrt((r-1)**2+(a-1)**2+(b-1)**2)

def nse(sim, obs):
    sim=np.asarray(sim,float); obs=np.asarray(obs,float)
    m=np.isfinite(sim)&np.isfinite(obs); sim,obs=sim[m],obs[m]
    if len(obs)<3: return np.nan
    return 1-((sim-obs)**2).sum()/((obs-obs.mean())**2).sum()

def metrics_block(sim, obs):
    sim=np.asarray(sim,float); obs=np.asarray(obs,float)
    m=np.isfinite(sim)&np.isfinite(obs); sim,obs=sim[m],obs[m]
    if len(obs)<3: return dict(n=len(obs),r=np.nan,bias=np.nan,mae=np.nan,rmse=np.nan,kge=np.nan,nse=np.nan)
    return dict(n=len(obs), r=np.corrcoef(sim,obs)[0,1], bias=(sim-obs).mean(),
                mae=np.abs(sim-obs).mean(), rmse=np.sqrt(((sim-obs)**2).mean()),
                kge=kge(sim,obs), nse=nse(sim,obs))

def contingency(sim, obs, thr):
    sim=np.asarray(sim,float); obs=np.asarray(obs,float)
    m=np.isfinite(sim)&np.isfinite(obs); sim,obs=sim[m],obs[m]
    fo=obs>=thr; fs=sim>=thr
    H=int((fo&fs).sum()); M=int((fo&~fs).sum()); F=int((~fo&fs).sum()); C=int((~fo&~fs).sum())
    n=H+M+F+C
    pod=H/(H+M) if (H+M) else np.nan
    far=F/(H+F) if (H+F) else np.nan
    csi=H/(H+M+F) if (H+M+F) else np.nan
    if n:
        acc=(H+C)/n; rand=((H+M)*(H+F)+(C+M)*(C+F))/n**2
        hss=(acc-rand)/(1-rand) if (1-rand) else np.nan
    else: hss=np.nan
    return dict(threshold=thr,H=H,M=M,F=F,C=C,POD=pod,FAR=far,CSI=csi,HSS=hss)

print('helpers ready')

BASE = /kaggle/input/datasets/neloypramanik4444/weather-dataset/dataset/
OUT  = /kaggle/working/
helpers ready


### Cell 2 — Load the seven datasets
Expect ~315k paired rows once stations meet ERA5. Focal-filled island stations are flagged, not dropped, at load.

In [2]:
bmd  = load('bmd', parse_dates=['date']).query("record_type == 'long'").copy()
era  = load('era5', parse_dates=['date']).copy()
meta = load('meta').copy()
ter  = load('terrain').copy()
skill= load('skill').copy()

# station data + coordinates
bmd = bmd.merge(meta[['station_name','lat','lon','elev_m','division','district','region']],
                on='station_name', how='left')

print('BMD long stations :', bmd.station_name.nunique())
print('ERA5 stations     :', era.station_name.nunique())
print('date span (BMD)   :', bmd.date.min().date(), '->', bmd.date.max().date())
assert bmd.station_name.nunique() == 36, 'expected 36 long stations'
assert set(bmd.station_name) <= set(era.station_name), 'station-name mismatch vs ERA5'
print('OK — 36 stations, names aligned with ERA5')

BMD long stations : 36
ERA5 stations     : 36
date span (BMD)   : 2000-01-01 -> 2023-12-31
OK — 36 stations, names aligned with ERA5


### Cell 3 — Homogeneity testing (Pettitt + SNHT) — *step 1.1*
We test each station's **annual-mean Tmax** and **annual-total precipitation** for a single change-point.
This documents series stability before any bias work. Pettitt is a rank-based non-parametric change-point
test; SNHT (Alexandersson) is a parametric shift test on standardised series. Both are self-contained here
(no extra packages).

In [3]:
def pettitt(x):
    x=np.asarray(x,float); x=x[np.isfinite(x)]; n=len(x)
    if n<10: return np.nan, np.nan
    # U_k statistic
    U=np.zeros(n)
    for k in range(n):
        U[k]=np.sum(np.sign(x[k].reshape(-1,1)-x[:]))  # placeholder, replaced below
    # vectorised Mann-Whitney style
    R=pd.Series(x).rank().values
    Uk=np.array([2*R[:k+1].sum()-(k+1)*(n+1) for k in range(n)])
    K=np.max(np.abs(Uk)); kloc=int(np.argmax(np.abs(Uk)))
    p=2*np.exp(-6*K**2/(n**3+n**2))
    return min(p,1.0), kloc

def snht(x):
    x=np.asarray(x,float); x=x[np.isfinite(x)]; n=len(x)
    if n<10: return np.nan, np.nan
    z=(x-x.mean())/x.std()
    T=np.full(n,np.nan)
    for a in range(1,n):
        z1=z[:a].mean(); z2=z[a:].mean()
        T[a]=a*z1**2+(n-a)*z2**2
    Tmax=np.nanmax(T); loc=int(np.nanargmax(T))
    # approximate 5% critical value for shift (Khaliq & Ouarda) ~ scales with ln(n)
    crit=4.44+0.5*np.log(n)   # rough; report statistic + flag
    return Tmax, loc, Tmax>crit

rows=[]
for s,g in bmd.groupby('station_name'):
    g=g.set_index('date')
    ann_tmax=g['tmax'].resample('YS').mean()
    ann_prcp=g['prcp'].resample('YS').sum(min_count=200)
    pt_p,_=pettitt(ann_tmax.values)
    sn=snht(ann_tmax.values); sn_stat=sn[0]; sn_flag=sn[2] if len(sn)==3 else np.nan
    pt_pr,_=pettitt(ann_prcp.values)
    rows.append(dict(station_name=s,
                     tmax_pettitt_p=round(pt_p,4),
                     tmax_snht_stat=round(sn_stat,2) if np.isfinite(sn_stat) else np.nan,
                     tmax_snht_break=bool(sn_flag) if sn_flag==sn_flag else np.nan,
                     prcp_pettitt_p=round(pt_pr,4)))
homog=pd.DataFrame(rows)
homog['tmax_homogeneous_5pct']=homog['tmax_pettitt_p']>0.05
n_break=(~homog['tmax_homogeneous_5pct']).sum()
print(f'Stations with Tmax change-point at 5% (Pettitt): {n_break} of 36')
homog.to_csv(OUT+'homogeneity_results.csv', index=False)
homog.head(10)

Stations with Tmax change-point at 5% (Pettitt): 20 of 36


,station_name,tmax_pettitt_p,tmax_snht_stat,tmax_snht_break,prcp_pettitt_p,tmax_homogeneous_5pct
0,Ambagan (Ctg),0.3439,6.21,True,0.8082,True
1,Barisal,0.0285,9.75,True,1.0000,False
2,Bhola,0.0014,14.93,True,0.5414,False
3,Bogra,0.0737,8.52,True,0.5671,True
4,Chandpur,0.0016,16.53,True,0.9256,False
5,Chittagong,0.0165,10.19,True,0.9624,False
6,Chuadanga,0.6205,6.03,True,0.0397,True
7,Comilla,0.3827,12.70,True,0.6766,True
8,Cox's Bazar,0.6766,5.18,False,0.3629,True
9,Dhaka,0.0067,11.76,True,0.2913,False


### Cell 4 — Pair stations to NASA POWER cells and ERA5-Land points — *step 1.2*
ERA5-Land is already extracted at station coordinates (36 points). For POWER (~0.5° grid, ~50 km),
we assign each station its nearest grid centre. Max nominal distance is ≈40 km, expected for this grid.

In [4]:
# --- ERA5 pairing (already point-extracted) ---
era_keep=['station_name','date','t2m_mean','t2m_max','t2m_min','prcp','swvl1','swvl2',
          'ssrd','pet_era5','aet_era5','ws2','ws10','sp','ea','es','rh_mean','vpd','era5_source']
paired_era5=(bmd.merge(era[era_keep], on=['station_name','date'], how='inner')
                .query("@STUDY_START <= date <= @STUDY_END").copy())
print('paired ERA5 rows :', len(paired_era5))

# --- POWER pairing (nearest cell) ---
power=load('power', parse_dates=['Date']).rename(columns={'Date':'date'})
cells=power[['LAT','LON']].drop_duplicates().reset_index(drop=True).values
def nearest_cell(lat,lon):
    d=(cells[:,0]-lat)**2+(cells[:,1]-lon)**2
    i=int(d.argmin()); return cells[i,0], cells[i,1], float(np.sqrt(d[i])*111)
meta[['pw_lat','pw_lon','pw_dist_km']]=meta.apply(
    lambda r: pd.Series(nearest_cell(r.lat,r.lon)), axis=1)
print('max station->POWER cell distance: %.1f km'%meta.pw_dist_km.max())

pw_ren={'T2M_MAX':'pw_tmax','T2M_MIN':'pw_tmin','T2M':'pw_tmean',
        'PRECTOTCORR':'pw_prcp','RH2M':'pw_rh','WS10M':'pw_ws10','GWETTOP':'pw_gwettop'}
power=power.rename(columns=pw_ren)
stn_cell=meta[['station_name','pw_lat','pw_lon','pw_dist_km']]
paired_power=(bmd.merge(stn_cell, on='station_name', how='left')
                 .merge(power, left_on=['pw_lat','pw_lon','date'],
                        right_on=['LAT','LON','date'], how='inner')
                 .query("@STUDY_START <= date <= @STUDY_END").copy())
print('paired POWER rows:', len(paired_power))

# persist station<->cell map (T2 support + reproducibility)
stn_cell.to_csv(OUT+'station_cell_map.csv', index=False)

paired ERA5 rows : 314326
max station->POWER cell distance: 40.8 km
paired POWER rows: 315058


### Cell 5 — Pooled validation, both products (T3) — *step 1.3*
Computes r, bias, MAE, RMSE, KGE, NSE for Tmax / Tmin / precipitation, for **NASA POWER and ERA5-Land**.
This is the headline benchmark table and the first time POWER is evaluated in this project.

In [5]:
def pooled_row(df, sim, obs, product, var):
    mb=metrics_block(df[sim], df[obs])
    mb.update(product=product, variable=var)
    return mb

T3=[]
# ERA5-Land
T3.append(pooled_row(paired_era5,'t2m_max','tmax','ERA5-Land','Tmax'))
T3.append(pooled_row(paired_era5,'t2m_min','tmin','ERA5-Land','Tmin'))
T3.append(pooled_row(paired_era5,'prcp_y','prcp_x' if 'prcp_x' in paired_era5 else 'prcp','ERA5-Land','Precip'))
# NASA POWER
T3.append(pooled_row(paired_power,'pw_tmax','tmax','NASA POWER','Tmax'))
T3.append(pooled_row(paired_power,'pw_tmin','tmin','NASA POWER','Tmin'))
T3.append(pooled_row(paired_power,'pw_prcp','prcp','NASA POWER','Precip'))

T3=pd.DataFrame(T3)[['product','variable','n','r','bias','mae','rmse','kge','nse']]
for c in ['r','bias','mae','rmse','kge','nse']: T3[c]=T3[c].round(3)
T3.to_csv(OUT+'T3_pooled_validation.csv', index=False)
print('=== TABLE T3 — Pooled validation (2000-2023) ===')
T3

=== TABLE T3 — Pooled validation (2000-2023) ===


,product,variable,n,r,bias,mae,rmse,kge,nse
0,ERA5-Land,Tmax,309833,0.896,-1.626,1.940,2.273,0.847,0.597
1,ERA5-Land,Tmin,308783,0.951,0.137,1.248,1.645,0.894,0.902
2,ERA5-Land,Precip,311423,0.448,-0.500,6.900,17.385,0.332,0.166
3,NASA POWER,Tmax,310418,0.781,-0.904,2.247,2.688,0.752,0.436
4,NASA POWER,Tmin,309368,0.943,0.110,1.413,1.831,0.927,0.879
5,NASA POWER,Precip,312155,0.515,0.462,6.722,17.640,0.491,0.144


### Cell 6 — Rainfall detection skill (T4) — *step 1.4*
Contingency-table scores at 1 / 10 / 20 / 50 mm for both products, **before correction**.
The 'after correction' columns are filled by NB3 once bias-corrected precipitation exists — here we
compute the *baseline* that the published POD ≤ 0.2 problem refers to.

In [6]:
def detection_table(df, sim, obs, product):
    out=[]
    for thr in RAIN_THRESHOLDS:
        c=contingency(df[sim], df[obs], thr); c.update(product=product, stage='raw')
        out.append(c)
    return out

prcp_obs_e = 'prcp_x' if 'prcp_x' in paired_era5 else 'prcp'
T4=[]
T4+=detection_table(paired_era5,'prcp_y',prcp_obs_e,'ERA5-Land')
T4+=detection_table(paired_power,'pw_prcp','prcp','NASA POWER')
T4=pd.DataFrame(T4)[['product','stage','threshold','H','M','F','C','POD','FAR','CSI','HSS']]
for c in ['POD','FAR','CSI','HSS']: T4[c]=T4[c].round(3)
T4.to_csv(OUT+'T4_rainfall_detection.csv', index=False)
print('=== TABLE T4 — Rainfall detection, RAW (before correction) ===')
print('Note POD collapse toward heavy thresholds — the documented <=0.2 problem.')
T4

=== TABLE T4 — Rainfall detection, RAW (before correction) ===
Note POD collapse toward heavy thresholds — the documented <=0.2 problem.


,product,stage,threshold,H,M,F,C,POD,FAR,CSI,HSS
0,ERA5-Land,raw,1,89736,9079,59625,152983,0.908,0.399,0.566,0.552
1,ERA5-Land,raw,10,28913,21873,32285,228352,0.569,0.528,0.348,0.411
2,ERA5-Land,raw,20,11300,20660,14837,264626,0.354,0.568,0.241,0.327
3,ERA5-Land,raw,50,1445,9301,2694,297983,0.134,0.651,0.108,0.178
4,NASA POWER,raw,1,90892,8156,57546,155561,0.918,0.388,0.580,0.571
5,NASA POWER,raw,10,32613,18317,29693,231532,0.640,0.477,0.405,0.483
6,NASA POWER,raw,20,15282,16777,16031,264065,0.477,0.512,0.318,0.424
7,NASA POWER,raw,50,2833,7958,4271,297093,0.263,0.601,0.188,0.297


### Cell 7 — Station inventory (T1) and data-source table (T2)

In [7]:
# completeness per station over the study window
span=pd.date_range(STUDY_START, STUDY_END, freq='D')
comp=[]
for s,g in bmd.query("@STUDY_START <= date <= @STUDY_END").groupby('station_name'):
    n_pos=len(span)
    comp.append(dict(station_name=s,
                     tmax_complete_pct=round(g['tmax'].notna().sum()/n_pos*100,1),
                     tmin_complete_pct=round(g['tmin'].notna().sum()/n_pos*100,1),
                     prcp_complete_pct=round(g['prcp'].notna().sum()/n_pos*100,1),
                     record_start=g['date'].min().date(), record_end=g['date'].max().date()))
comp=pd.DataFrame(comp)

T1=(meta[['station_name','wmo_id','lat','lon','elev_m','division','district']]
      .merge(comp, on='station_name', how='left')
      .merge(skill[['station_name','land_frac_9km','elev_std_9km']], on='station_name', how='left'))
T1['focal_filled']=T1['station_name'].isin(FOCAL_FILLED)
T1=T1.round({'land_frac_9km':3,'elev_std_9km':2,'lat':3,'lon':3})
T1.to_csv(OUT+'T1_station_inventory.csv', index=False)
print('=== TABLE T1 — Station inventory (head) ===')
print(T1.head(8).to_string(index=False))

# T2 — data sources (static descriptor; POWER schema reflects the ACTUAL file)
T2=pd.DataFrame([
 dict(source='BMD stations', product='In-situ daily', resolution='36 points',
      period='2000-2023', variables='tmax, tmin, prcp',
      access='Bangladesh Meteorological Dept (cleaned panel)'),
 dict(source='NASA POWER', product='Reanalysis/satellite blend', resolution='0.5 deg (~50 km)',
      period='2000-2026', variables='T2M, T2M_MAX, T2M_MIN, PRECTOTCORR, RH2M, WS10M, GWETTOP',
      access='NASA POWER API (larc.nasa.gov)'),
 dict(source='ERA5-Land', product='Reanalysis (CHTESSEL)', resolution='0.1 deg (~9 km)',
      period='2000-2026', variables='t2m, d2m, prcp, swvl1/2, ssrd, sp, u10/v10 (derived ws2, ea/es, rh, vpd)',
      access='GEE ECMWF/ERA5_LAND/DAILY_AGGR at station coords'),
 dict(source='Terrain', product='DEM + surface water', resolution='30 m aggregated to 9 km',
      period='static', variables='land_frac_9km, elev_std_9km, roughness, tpi, dist_water, slope/aspect',
      access='Copernicus GLO-30 + SRTM v3 + JRC GSW via GEE'),
])
T2.to_csv(OUT+'T2_data_sources.csv', index=False)
print('\nT2 written.')

=== TABLE T1 — Station inventory (head) ===
 station_name  wmo_id   lat    lon  elev_m   division   district  tmax_complete_pct  tmin_complete_pct  prcp_complete_pct record_start record_end  land_frac_9km  elev_std_9km  focal_filled
Ambagan (Ctg)     NaN 22.36 91.790    34.0 Chattogram Chattogram               95.0               90.9               93.4   2000-01-01 2023-12-31          0.951         14.10         False
      Barisal 41950.0 22.75 90.367     4.0   Barishal   Barishal               99.9               95.8               99.9   2000-01-01 2023-12-31          0.933          3.58         False
        Bhola 41951.0 22.68 90.650     4.0   Barishal      Bhola               97.6               99.9               99.9   2000-01-01 2023-12-31          0.928          4.00         False
        Bogra 41883.0 24.85 89.370    20.0   Rajshahi     Bogura               99.4               99.6              100.0   2000-01-01 2023-12-31          1.000          3.00         False
     Chandp

### Cell 8 — Persist paired working tables for NB2–NB4
These parquet files are the inputs to the terrain analysis, bias correction, and reconstruction notebooks.
Keeping them as parquet preserves dtypes and is fast to reload.

In [8]:
# harmonise the ERA5 precip column name for downstream clarity
if 'prcp_x' in paired_era5.columns:
    paired_era5=paired_era5.rename(columns={'prcp_x':'obs_prcp','prcp_y':'era5_prcp'})
else:
    paired_era5=paired_era5.rename(columns={'prcp':'obs_prcp'})
paired_era5=paired_era5.rename(columns={'tmax':'obs_tmax','tmin':'obs_tmin'})

paired_era5.to_parquet(OUT+'paired_era5.parquet', index=False)
paired_power.to_parquet(OUT+'paired_power.parquet', index=False)
meta.to_csv(OUT+'meta_with_cells.csv', index=False)

print('Wrote for NB2-NB4:')
for f in ['paired_era5.parquet','paired_power.parquet','station_cell_map.csv',
          'meta_with_cells.csv','T1_station_inventory.csv','T2_data_sources.csv',
          'T3_pooled_validation.csv','T4_rainfall_detection.csv','homogeneity_results.csv']:
    print('  ', OUT+f)
print('\nNB1 complete. Next: run Notebook 2 (terrain controls + statistical bias correction).')

Wrote for NB2-NB4:
   /kaggle/working/paired_era5.parquet
   /kaggle/working/paired_power.parquet
   /kaggle/working/station_cell_map.csv
   /kaggle/working/meta_with_cells.csv
   /kaggle/working/T1_station_inventory.csv
   /kaggle/working/T2_data_sources.csv
   /kaggle/working/T3_pooled_validation.csv
   /kaggle/working/T4_rainfall_detection.csv
   /kaggle/working/homogeneity_results.csv

NB1 complete. Next: run Notebook 2 (terrain controls + statistical bias correction).
